In [1]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from bs4 import BeautifulSoup
import time, csv
from urllib.parse import urljoin

START_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
}

r = requests.get(START_URL, headers=HEADERS, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

print("状态码:", r.status_code)
print("网页标题:", soup.title.get_text(strip=True) if soup.title else None)
rows = soup.select("div.property")
print("本页房源条数:", len(rows))

状态码: 200
网页标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
本页房源条数: 60


In [2]:
row = rows[0]

infos = row.select("p.property-content-info-text")
print("属性组原始内容:")
for i, t in enumerate(infos):
    print(i, "|", t.get_text(strip=True))

print()
print("title   :", row.select_one("h3.property-content-title-name").get_text(strip=True))
print("house   :", row.select_one("p.property-content-info-comm-name").get_text(strip=True))
print("address :", row.select_one("p.property-content-info-comm-address").get_text(strip=True))
print("price总价:", row.select_one("p.property-price-total").get_text(strip=True))
print("price单价:", row.select_one("p.property-price-average").get_text(strip=True))

属性组原始内容:
0 | 3室2厅1卫
1 | 89.9㎡
2 | 南北
3 | 低层(共27层)
4 | 2019年建造

title   : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house   : 万科金域学府翰林菁华里
address : 南岸学府大道二塘路59号
price总价: 148万
price单价: 16465元/㎡


In [3]:
def parse_one(row):
    def text(sel, default=""):
        node = row.select_one(sel)
        return node.get_text(strip=True) if node else default

    title = text("h3.property-content-title-name")
    house = text("p.property-content-info-comm-name")
    address = text("p.property-content-info-comm-address")
    price = text("p.property-price-average")

    infos = row.select("p.property-content-info-text")
    vals = [t.get_text(strip=True) for t in infos]

    struct   = vals[0] if len(vals) > 0 else ""  
    area     = vals[1] if len(vals) > 1 else ""  
    # vals[2] 是朝向，本实验不采集
    floor    = vals[3] if len(vals) > 3 else ""  
    maketime = vals[4] if len(vals) > 4 else ""   

    return [title, house, address, struct, area, floor, maketime, price]

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

# 拿前三条试一下
for row in rows[:3]:
    print(parse_one(row))

['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']
['黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '渝北渝北黄泥磅紫荆路29号', '3室2厅2卫', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡']
['鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '渝北鸳鸯龙展路59号', '3室2厅1卫', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡']


In [4]:
nxt = soup.select_one("a.next.next-active")
print("节点:", nxt)
print("文字:", nxt.get_text(strip=True) if nxt else None)
print("href:", nxt.get("href") if nxt else None)
print("补全后:", urljoin(START_URL, nxt.get("href")) if nxt else None)

节点: <a class="next next-active" data-npv="" data-v-29d65fc6="" href="?page=2">下一页</a>
文字: 下一页
href: ?page=2
补全后: http://www.bspider.top/anjuke/?page=2


In [5]:
records = []
url = START_URL
visited = set()

for page_no in range(1, 3):          # 先只跑 2 页
    print("正在采集第", page_no, "页:", url)
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    page = BeautifulSoup(resp.text, "html.parser")

    items = page.select("div.property")
    if not items:
        print("本页没有房源，停止")
        break
    for row in items:
        records.append(parse_one(row))
    print("  本页", len(items), "条，累计", len(records), "条")

    nxt = page.select_one("a.next.next-active")
    if nxt is None:
        print("  已到末页（没有下一页按钮）")
        break
    nxt_url = urljoin(url, nxt.get("href"))
    if not nxt_url.startswith("http://www.bspider.top/"):   # 不许跳出教学站
        print("  下一页地址越界，停止:", nxt_url)
        break
    if nxt_url in visited:
        print("  下一页重复，停止")
        break
    visited.add(url)
    url = nxt_url
    time.sleep(1)                    # 至少等 1 秒，别把人家站点打爆

page1_titles = set()
resp = requests.get(START_URL, headers=HEADERS, timeout=30)
resp.encoding = resp.apparent_encoding
page1_titles = {t.get_text(strip=True)
                for t in BeautifulSoup(resp.text, "html.parser").select("h3.property-content-title-name")}
page2_titles = {rec[0] for rec in records[60:120]}
print("第一页标题数:", len(page1_titles))
print("第二页与第一页重复的条数:", len(page1_titles & page2_titles), "（应为 0）")

正在采集第 1 页: http://www.bspider.top/anjuke/
  本页 60 条，累计 60 条
正在采集第 2 页: http://www.bspider.top/anjuke/?page=2
  本页 60 条，累计 120 条
第一页标题数: 60
第二页与第一页重复的条数: 0 （应为 0）


In [6]:
records = []
url = START_URL
visited = set()

for page_no in range(1, 20):        # 安全上限，防止死循环
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    page = BeautifulSoup(resp.text, "html.parser")
    items = page.select("div.property")
    if not items:
        print("第", page_no, "页没有房源，停止")
        break
    for row in items:
        records.append(parse_one(row))
    print("第", page_no, "页", url, "本页", len(items), "条，累计", len(records), "条")

    nxt = page.select_one("a.next.next-active")
    if nxt is None:
        print("第", page_no, "页后没有下一页按钮 → 末页，采集结束")
        break
    nxt_url = urljoin(url, nxt.get("href"))
    if not nxt_url.startswith("http://www.bspider.top/") or nxt_url in visited:
        print("地址越界或重复，停止")
        break
    visited.add(url)
    url = nxt_url
    time.sleep(1)

print("采集结束，总条数:", len(records))

第 1 页 http://www.bspider.top/anjuke/ 本页 60 条，累计 60 条
第 2 页 http://www.bspider.top/anjuke/?page=2 本页 60 条，累计 120 条
第 3 页 http://www.bspider.top/anjuke/?page=3 本页 60 条，累计 180 条
第 4 页 http://www.bspider.top/anjuke/?page=4 本页 60 条，累计 240 条
第 5 页 http://www.bspider.top/anjuke/?page=5 本页 60 条，累计 300 条
第 6 页 http://www.bspider.top/anjuke/?page=6 本页 60 条，累计 360 条
第 7 页 http://www.bspider.top/anjuke/?page=7 本页 60 条，累计 420 条
第 8 页 http://www.bspider.top/anjuke/?page=8 本页 60 条，累计 480 条
第 9 页 http://www.bspider.top/anjuke/?page=9 本页 60 条，累计 540 条
第 10 页 http://www.bspider.top/anjuke/?page=10 本页 60 条，累计 600 条
第 10 页后没有下一页按钮 → 末页，采集结束
采集结束，总条数: 600


In [7]:
import os
from pathlib import Path

out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

before = len(records)
seen = set()
unique = []
for rec in records:
    key = tuple(rec)              # 按 8 个字段完全相同判定重复
    if key not in seen:
        seen.add(key)
        unique.append(rec)

print("去重前:", before, "条；去重后:", len(unique), "条；去掉:", before - len(unique), "条")

with open(out_dir / "houseinfo.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(FIELDS)
    w.writerows(unique)
print("已保存 data/houseinfo.csv")

去重前: 600 条；去重后: 498 条；去掉: 102 条
已保存 data/houseinfo.csv


In [8]:
import csv as _csv
with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    rd = list(_csv.DictReader(f))

print("表头:", rd[0].keys())
print("数据行数（不含表头）:", len(rd))
for rec in [rd[0], rd[len(rd)//2], rd[-1]]:      # 第一条、中间一条、最后一条
    print("-" * 60)
    for k in FIELDS:
        print(f"{k:9s}: {rec[k]}")

表头: dict_keys(['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price'])
数据行数（不含表头）: 498
------------------------------------------------------------
title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    : 万科金域学府翰林菁华里
address  : 南岸学府大道二塘路59号
struct   : 3室2厅1卫
area     : 89.9㎡
floor    : 低层(共27层)
maketime : 2019年建造
price    : 16465元/㎡
------------------------------------------------------------
title    : 精装修洋房 位置安静 全新装修 随时看房
house    : 协信城立方玺院
address  : 沙坪坝大学城大学城南二路
struct   : 3室3厅2卫
area     : 95.8㎡
floor    : 中层(共8层)
maketime : 2016年建造
price    : 15658元/㎡
------------------------------------------------------------
title    : 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售
house    : 恒大照母山山水城
address  : 渝北照母山恒康路9号
struct   : 3室2厅1卫
area     : 90.2㎡
floor    : 低层(共32层)
maketime : 2015年建造
price    : 15530元/㎡


In [1]:
import csv

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("CSV 表头:", reader.fieldnames)
    assert reader.fieldnames == FIELDS, "表头和8字段顺序不一致，先回去检查第二阶段！"
    records = [[r[k].strip() for k in FIELDS] for r in reader]

print("CSV 数据条数:", len(records))
print("第一条:", records[0])

CSV 表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据条数: 498
第一条: ['万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡']


In [2]:
import pymysql

conn = pymysql.connect(
    host="mysql", port=3306,
    database="smartpark_lab",
    user="student", password="123456",
    charset="utf8mb4",
    autocommit=False,        # 由程序 commit，手册要求
    connect_timeout=5,
)
cur = conn.cursor()

cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
print(cur.fetchone())

('8.4.11', 'smartpark_lab', 'student@%')


In [3]:
cur.execute("SHOW TABLES")
print("库里的表:", cur.fetchall())

cur.execute("SELECT COUNT(*) FROM houseinfo")
before = cur.fetchone()[0]
print("入库前 houseinfo 条数:", before)

库里的表: (('houseinfo',),)
入库前 houseinfo 条数: 0


In [4]:
cur.execute(
    "SELECT title,house,address,struct,area,floor,maketime,price FROM houseinfo"
)
existing = set(cur.fetchall())
print("库内已有记录数:", len(existing))

new_records = [tuple(r) for r in records if tuple(r) not in existing]
print("待新增记录数:", len(new_records))

库内已有记录数: 0
待新增记录数: 498


In [5]:
sql = ("INSERT INTO houseinfo "
       "(title,house,address,struct,area,floor,maketime,price) "
       "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)")

try:
    cur.executemany(sql, new_records[:3])   # 只插前3条试试
    conn.commit()                            # 成功才提交
    print("试写 3 条成功")
    cur.execute("SELECT id,title,struct,area,price FROM houseinfo ORDER BY id DESC LIMIT 3")
    for row in cur.fetchall():
        print(row)
except Exception as e:
    conn.rollback()                          # 出错撤销本次未提交的修改
    print("出错已回滚:", e)

试写 3 条成功
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '3室2厅1卫', '88㎡', '16487元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '3室2厅2卫', '94.5㎡', '14815元/㎡')
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '3室2厅1卫', '89.9㎡', '16465元/㎡')


In [6]:
try:
    cur.executemany(sql, new_records)        # 剩余的全部插入
    conn.commit()
    print("全量入库成功")
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后总条数:", cur.fetchone()[0])
except Exception as e:
    conn.rollback()
    print("出错已回滚:", e)
finally:
    cur.close()                              # 先关游标
    conn.close()                             # 再关连接
    print("游标和连接已关闭")

全量入库成功
入库后总条数: 501
游标和连接已关闭


In [7]:
conn = pymysql.connect(host="mysql", port=3306, database="smartpark_lab",
                       user="student", password="123456",
                       charset="utf8mb4", autocommit=False, connect_timeout=5)
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM houseinfo")
print("当前总条数:", cur.fetchone()[0])

cur.execute("SELECT title,house,address,struct,area,floor,maketime,price FROM houseinfo")
existing = set(cur.fetchall())
new_records = [tuple(r) for r in records if tuple(r) not in existing]
print("再次运行时待新增:", len(new_records), "（应为 0）")

cur.close()
conn.close()

当前总条数: 501
再次运行时待新增: 0 （应为 0）
